<a href="https://colab.research.google.com/github/priyal6/AI-AGENT/blob/main/test_eval_smith.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install --upgrade langsmith

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 359.9/359.9 kB 14.1 MB/s eta 0:00:00
  Attempting uninstall: langsmith
    Found existing installation: langsmith 0.7.18
    Uninstalling langsmith-0.7.18:
      Successfully uninstalled langsmith-0.7.18


In [ ]:
from google.colab import userdata
api_key = userdata.get('OPENAI_API_KEY')

In [ ]:
import os
from google.colab import userdata

os.environ['LANGSMITH_TRACING'] = 'true'
os.environ['LANGSMITH_ENDPOINT'] = 'https://eu.api.smith.langchain.com'
os.environ['LANGSMITH_API_KEY'] = userdata.get('LANGSMITH_API_KEY')
os.environ['LANGSMITH_PROJECT'] = 'pr-clear-muscle-85'
os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')

In [ ]:
from langsmith import traceable, wrappers
from openai import OpenAI

client = wrappers.wrap_openai(OpenAI(api_key=api_key))

@traceable
def retrieve_docs(question: str) -> str:
    return "Overfitting happens when a model learns noise from training data."

@traceable
def build_messages(question: str, context: str):
    return [
        {"role": "system", "content": f"Answer using this context only:\n{context}"},
        {"role": "user", "content": question},
    ]

@traceable(run_type="llm", name="OpenAI Answer Call")
def call_llm(messages):
    response = client.chat.completions.create(
        model="gpt-4.1-mini",
        messages=messages,
    )
    return response.choices[0].message.content

@traceable(name="RAG Pipeline")
def answer_question(question: str) -> str:
    context = retrieve_docs(question)
    messages = build_messages(question, context)
    return call_llm(messages)

print(answer_question("What is overfitting?"))


Overfitting happens when a model learns noise from the training data.


In [ ]:
datagen_model = "gpt-4o-mini"
output_string = ""
for i in range(3):
  question = f"""
  I am creating input output training pairs to fine tune my gpt model. The usecase is a retailer generating a description for a product from a product catalogue. I want the input to be product name and category (to which the product belongs to) and output to be description.
  The format should be of the form:
  1.
  Input: product_name, category
  Output: description
  2.
  Input: product_name, category
  Output: description

  Do not add any extra characters around that formatting as it will make the output parsing break.
  Create as many training pairs as possible.
  """

  response = client.chat.completions.create(
    model=datagen_model,
    messages=[
      {"role": "system", "content": "You are a helpful assistant designed to generate synthetic data."},
      {"role": "user", "content": question}
    ]
  )
  res = response.choices[0].message.content
  output_string += res + "\n" + "\n"
print(output_string[:3000])

1.
Input: Smartwatch Series 6, Electronics
Output: The Smartwatch Series 6 combines cutting-edge technology with sleek style, featuring fitness tracking, heart rate monitoring, and customizable watch faces to suit any occasion.

2.
Input: Organic Green Tea, Food & Beverages
Output: Our Organic Green Tea is harvested from the finest leaves, providing a refreshing taste and numerous health benefits, including antioxidants and improved metabolism.

3.
Input: Bluetooth Noise Cancelling Headphones, Electronics
Output: Experience immersive sound with our Bluetooth Noise Cancelling Headphones, designed to block out ambient noise while delivering crystal-clear audio for an unparalleled listening experience.

4.
Input: Yoga Mat, Sports & Outdoors
Output: Enhance your yoga practice with our high-quality Yoga Mat, featuring a non-slip surface for stability and comfort, perfect for both beginners and seasoned yogis alike.

5.
Input: Stainless Steel Water Bottle, Sports & Outdoors
Output: Stay hydr

In [ ]:
import os
from langsmith import Client, traceable
from langsmith.evaluation import evaluate
from google.colab import userdata


client = Client()

dataset=client.create_dataset(
    dataset_name="Mini QA Dataset",
    description="Small demo dataset for Langsmith evaluation"
)

client.create_examples(
    dataset_id=dataset.id,
    examples=[
        {
            "inputs": {"question": "What is overfitting?"},
            "outputs": {
                "answer": "Overfitting is when a model learns noise or patterns specific to the training data and fails to generalize well."
            },
        },
        {
            "inputs": {"question": "What is underfitting?"},
            "outputs": {
                "answer": "Underfitting is when a model is too simple to capture the underlying patterns in the data."
            },
        },
    ],
)

@traceable(name="Retrieve Docs")
def retrieve_docs(question: str) -> str:
  kb = {
      "overfitting": "Overfitting happens when a model memorizes noise in the training data and performs poorly on unseen data.",
      "underfitting": "Underfitting happens when a model is too simple and cannot learn the real pattern in the data."
  }
  q = question.lower()
  if "overfitting" in q:
    return kb["overfitting"]
  if "underfitting" in q:
    return kb["underfitting"]
  return "No relevant context found."


@traceable(name="Generated Answer")
def generate_answer(question: str, context: str) -> str:
  return context

@traceable(name="Mini RAG Test App")
def app(inputs: dict) -> dict:
  question = inputs["question"]
  context = retrieve_docs(question)
  answer = generate_answer(question, context)
  return {"answer": answer}

def exact_match(output: dict, reference_output: dict) -> bool:
  pred = output["answer"].strip().lower()
  ref = reference_output["answer"].strip().lower()
  return pred == ref

results = evaluate(
    app,
    data= dataset.name,
    evaluators = [exact_match],
    experiment_prefix = "mini-rag-experiment"
)

print(results)

View the evaluation results for experiment: 'mini-rag-experiment-65ea15dc' at:
https://eu.smith.langchain.com/o/ee50d027-04ad-41a1-a50f-cfa1a716c955/datasets/f550ffdc-8fa2-417f-bd73-db8603bfef0f/compare?selectedSessions=eaff21c5-e1c3-4d05-9e8e-5650fff0b428




0it [00:00, ?it/s]

ERROR:langsmith.evaluation._runner:Error running evaluator <DynamicRunEvaluator exact_match> on run 019d2c0f-22e1-7473-bd01-ec38ff4ff1a0: TypeError("'RunTree' object is not subscriptable")
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/langsmith/evaluation/_runner.py", line 1637, in _run_evaluators
    evaluator_response = evaluator.evaluate_run(  # type: ignore[call-arg]
                         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/langsmith/evaluation/evaluator.py", line 332, in evaluate_run
    result = self.func(
             ^^^^^^^^^^
  File "/tmp/ipykernel_17583/2753375721.py", line 58, in exact_match
    pred = output["answer"].strip().lower()
           ~~~~~~^^^^^^^^^^
TypeError: 'RunTree' object is not subscriptable
ERROR:langsmith.evaluation._runner:Error running evaluator <DynamicRunEvaluator exact_match> on run 019d2c0f-2314-7043-98a3-483b9ec33619: TypeError("'RunTree' object 

<ExperimentResults mini-rag-experiment-65ea15dc>


In [ ]:
# pytest

!pip install -U "langsmith[pytest]"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.4/42.4 kB 2.6 MB/s eta 0:00:00


In [ ]:
import os
from langsmith import traceable, wrappers
import openai

os.environ["LANGSMITH_TEST_SUITE"] = "SQL app tests"

oai_client = wrappers.wrap_openai(openai.OpenAI())

@traceable
def generate_sql(user_query: str) -> str:
    result = oai_client.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": "Convert the user query to a SQL query"},
            {"role": "user", "content": user_query},
        ]
    )
    return result.choices[0].message.content

def is_valid_sql(sql_query: str) -> float:
    return 1.0 if sql_query.strip() else 0.0

In [ ]:
%%writefile test_sql_app.py
import os
import pytest
from langsmith import traceable, wrappers
import openai
from langsmith import testing as t

os.environ["LANGSMITH_TEST_SUITE"] = "SQL app tests"

oai_client = wrappers.wrap_openai(openai.OpenAI())

@traceable
def generate_sql(user_query: str) -> str:
    result = oai_client.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": "Convert the user query to a SQL query"},
            {"role": "user", "content": user_query},
        ]
    )
    content = result.choices[0].message.content

    if content.startswith('```sql') and content.endswith('```'):
        return content[7:-3].strip()
    return content

def is_valid_sql(sql_query: str) -> float:
    """Return True if the query is valid SQL."""
    return True

@pytest.mark.langsmith
def test_sql_generation_select_all() -> None:
    user_query = "Get all users from the customers table"
    t.log_inputs({"user_query": user_query})
    expected = "SELECT * FROM customers;"
    t.log_reference_outputs({"sql": expected})

    sql = generate_sql(user_query)
    t.log_outputs({"sql": sql})

    t.log_feedback(key="valid_sql", score=is_valid_sql(sql))
    assert sql == expected

Overwriting test_sql_app.py


In [ ]:
!pytest -v test_sql_app.py

============================= test session starts ==============================
platform linux -- Python 3.12.13, pytest-8.4.2, pluggy-1.6.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content
plugins: langsmith-0.7.22, anyio-4.12.1, typeguard-4.5.1
collected 1 item                                                               

test_sql_app.py::test_sql_generation_select_all PASSED                   [100%]

============================== 1 passed in 4.19s ===============================
